# 🚀 HackAIthon 2026 - Advanced Pipeline (Gemma-4 + LangGraph + Unsloth)

In [ ]:
!pip uninstall -y vllm 2>/dev/null
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install langgraph pydantic nest_asyncio sympy z3-solver numpy scipy pandas tqdm


## Bước 1: Import và Môi trường

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import re
import json
import subprocess
import tempfile
import sys
import logging
import torch
import gc
from typing import TypedDict, Optional, List

# --- XÁC THỰC HUGGING FACE CHO COLAB ---
# Yêu cầu: Bạn phải tạo một Secret tên là HF_TOKEN trong menu Chìa khóa bên trái của Colab
try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Đăng nhập Hugging Face thành công!")
except Exception as e:
    print("Chưa cấu hình HF_TOKEN trong Secrets hoặc không chạy trên Colab.")
# ---------------------------------------
from langgraph.graph import StateGraph, END
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Cấu hình Mô hình
LLM_MODEL = "google/gemma-4-E4B-it"

class AnswerModel(BaseModel):
    reasoning: str
    answer: str

LETTER_MAP = {i: chr(65 + i) for i in range(26)}
def format_choices(choices: list) -> str:
    return "\n".join(f"{LETTER_MAP[i]}. {c}" for i, c in enumerate(choices))



## Bước 2: Tiện ích Executor

In [ ]:
def extract_python_code(text: str) -> str:
    match = re.search(r'```python\n(.*?)\n```', text, re.DOTALL)
    if match: return match.group(1).strip()
    return text.strip()

def execute_python_code(code: str, timeout: int = 10) -> tuple[bool, str]:
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False, encoding='utf-8') as tmp_file:
        tmp_file.write(code)
        tmp_path = tmp_file.name
    try:
        limit_code = "import resource, sys\ntry:\n    resource.setrlimit(resource.RLIMIT_AS, (2 * 1024**3, 2 * 1024**3))\nexcept:\n    pass\n"
        with open(tmp_path, 'w', encoding='utf-8') as f:
            f.write(limit_code + code)
        result = subprocess.run([sys.executable, tmp_path], capture_output=True, text=True, timeout=timeout)
        if result.returncode == 0: return True, result.stdout.strip()
        return False, result.stderr.strip()
    except Exception as e:
        return False, str(e)
    finally:
        if os.path.exists(tmp_path): os.remove(tmp_path)



## Bước 3: Khởi tạo Models (Chạy 1 lần trên Kaggle)

In [ ]:
from unsloth import FastLanguageModel
import torch

logger.info(f"Đang tải {LLM_MODEL} bằng Unsloth (4-bit) vào T4 GPU...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LLM_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def generate_text(prompt_text, max_tokens=1024):
    msgs = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[formatted], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_tokens, use_cache=True, temperature=0.1)
    # Cắt bỏ phần prompt để lấy đúng câu trả lời
    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.batch_decode(outputs[:, input_length:], skip_special_tokens=True)[0]
    return response



## Bước 4: Định nghĩa Nodes LangGraph

In [ ]:
class AgentState(TypedDict):
    qid: str
    question: str
    choices: List[str]
    question_type: Optional[str]
    python_code: Optional[str]
    python_result: Optional[str]
    python_success: bool
    retry_count: int
    reasoning: Optional[str]
    final_answer: Optional[str]

SAFETY_PATTERNS = [
    r"làm thế nào để.*trái", r"cách.*phá hoại", r"làm giả",
    r"xúc phạm.*biểu tượng", r"xúc phạm.*lãnh tụ", r"lợi dụng.*tham nhũng",
    r"phát tán.*mật", r"làm suy yếu", r"chống lại đường lối",
    r"kích động.*chia rẽ", r"phá hoại.*tinh thần", r"tránh.*bị phát hiện"
]

CODEABLE_PATTERNS = [
    r"\\$", r"\\frac", r"\\sqrt", r"\\int", r"\\sum", r"\\lim",
    r"\\log", r"\\sin", r"\\cos", r"\\tan", r"\\text",
    r"tính\\b", r"giải\\b", r"phương trình", r"bất phương trình",
    r"hệ phương trình", r"biểu thức", r"đạo hàm", r"tích phân",
    r"ước\\b", r"bội\\b", r"số nguyên tố", r"ưcln", r"bcnn",
    r"xác suất", r"vận tốc", r"gia tốc", r"lực\\b", r"điện trở",
    r"công suất", r"năng lượng", r"bước sóng", r"mol\\b", r"dung dịch", r"ph"
]

NON_CODEABLE_PATTERNS = [
    r"đoạn thông tin", r"đoạn văn", r"đoạn trích", r"văn bản", r"bài đọc",
    r"tác giả", r"nhân vật", r"nội dung", r"ý nghĩa", r"biện pháp tu từ",
    r"phong cách", r"thể loại", r"chủ đề"
]

def node_router(state: AgentState) -> AgentState:
    logger.info(f"[{state['qid']}] Node 1: Classifier")
    text_lower = state['question'].lower()
    
    if any(re.search(p, text_lower) for p in SAFETY_PATTERNS):
        state["question_type"] = "SAFETY"
    elif any(re.search(p, text_lower) for p in NON_CODEABLE_PATTERNS) or len(text_lower) > 2000:
        state["question_type"] = "QA"
    elif any(re.search(p, text_lower) for p in CODEABLE_PATTERNS):
        state["question_type"] = "CODEABLE"
    else:
        state["question_type"] = "QA" 
    return state

def node_coder(state: AgentState) -> AgentState:
    logger.info(f"[{state['qid']}] Node 2: Coder (Retry: {state.get('retry_count', 0)})")
    
    if state.get("retry_count", 0) == 0:
        prompt = f"""Bạn là một chuyên gia lập trình Python tối giản, phụ trách xử lý các bài toán định lượng và logic đa ngành.
Nhiệm vụ: Viết mã nguồn Python để giải quyết câu hỏi dưới đây.

QUY TẮC ÉP KHUÔN ĐỊNH DẠNG TUYỆT ĐỐI:
1. CẤM TUYỆT ĐỐI SỬ DỤNG KÝ TỰ DẤU THĂNG (#). Không viết comment.
2. LỆNH PRINT CỐT LÕI: Lệnh print cuối cùng BẮT BUỘC chỉ in ra GIÁ TRỊ SỐ NGUYÊN HOẶC SỐ THỰC THUẦN TÚY (VD: print(float(kq))) hoặc kết quả biểu thức sạch (str(kq)).
3. KHÔNG in chữ cái đáp án A, B, C, D.
4. Toàn bộ mã nguồn phải bọc trong khối try-except toàn cục, nhánh except in ra 'ERROR'.
5. Chỉ xuất mã nguồn nằm gọn trong khối ```python và ```.

Câu hỏi: {state['question']}"""
    else:
        prompt = f"""Bạn là chuyên gia sửa lỗi Python. Mã giải toán trước đó đã thất bại.
Câu hỏi: {state['question']}

Mã Python bị lỗi:
```python
{state['python_code']}
```
Lỗi trả về:
{state['python_result']}

Nhiệm vụ: Sửa mã nguồn để giải quyết lỗi trên. Vẫn phải tuân thủ tuyệt đối quy tắc bọc try-except và in ra kết quả số thuần túy.
Chỉ xuất mã trong khối ```python và ```."""

    raw_response = generate_text(prompt, max_tokens=1024)
    
    code = extract_python_code(raw_response)
    state["python_code"] = code
    success, result = execute_python_code(code)
    state["python_result"] = result
    state["python_success"] = success
    
    if not success and state.get("retry_count", 0) < 1:
        state["retry_count"] = state.get("retry_count", 0) + 1
    return state

def node_instructor(state: AgentState) -> AgentState:
    logger.info(f"[{state['qid']}] Node 3: Instructor")
    choices_text = format_choices(state['choices'])
    valid_letters_list = [LETTER_MAP[i] for i in range(len(state['choices']))]
    valid_letters_str = ", ".join(valid_letters_list)
    
    if state["question_type"] == "CODEABLE":
        prompt = f"""Bạn là một giám khảo AI. Hãy chọn đáp án chính xác dựa trên kết quả tính toán của đoạn mã Python cung cấp dưới đây.

---
Ví dụ 1:
Question: Tính diện tích hình tròn bán kính r = 3m. (lấy pi = 3.14)
Choices:
A. 28.26 m2
B. 9.42 m2
C. 18.84 m2
[Kết quả chạy code Python]: 28.259999999
Reasoning: Mã Python in ra kết quả là 28.259999999, giá trị này trùng khớp với 28.26 m2 (Lựa chọn A) do sai số dấu phẩy động. Vì vậy, đáp án đúng là A.
Answer: A
---

Câu hỏi: {state['question']}
Lựa chọn:
{choices_text}

[Kết quả chạy code Python]:
{state.get('python_result', 'N/A')}

Lưu ý: Nếu kết quả Python bị lỗi, hãy tự suy luận để giải quyết. Chỉ chọn MỘT đáp án từ: {valid_letters_str}."""

    else:
        prompt = f"""Bạn là một trợ lý AI thông thái. Hãy đọc kỹ và trả lời câu hỏi trắc nghiệm sau.

---
Ví dụ 1:
Question: Thủ đô của Việt Nam là gì?
Choices:
A. TP. Hồ Chí Minh
B. Hà Nội
C. Đà Nẵng
D. Hải Phòng
Reasoning: Hà Nội là thủ đô của nước Cộng hòa Xã hội Chủ nghĩa Việt Nam theo hiến pháp. Vì vậy, đáp án đúng là B.
Answer: B
---

Câu hỏi: {state['question']}
Lựa chọn:
{choices_text}

Lưu ý: Chỉ chọn MỘT đáp án từ: {valid_letters_str}."""

    json_instruction = """
YÊU CẦU BẮT BUỘC: Bạn CHỈ ĐƯỢC PHÉP trả về duy nhất một đối tượng JSON hợp lệ, tuyệt đối không viết thêm bất kỳ văn bản nào khác. Định dạng JSON phải chính xác như sau:
{
    "reasoning": "Giải thích ngắn gọn lý do chọn đáp án này",
    "answer": "Một ký tự đáp án (ví dụ: A)"
}
"""
    prompt += json_instruction

    raw_response = generate_text(prompt, max_tokens=512)
    try:
        
        json_match = re.search(r'\{.*?\}', raw_response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group(0))
        else:
            parsed = json.loads(raw_response)
        
        state["reasoning"] = parsed.get("reasoning", "")
        ans = parsed.get("answer", "A").strip()
        state["final_answer"] = ans if ans in valid_letters_list else "A"
    except:
        state["reasoning"] = raw_response
        state["final_answer"] = "A"

    return state



## Bước 5: Biên dịch LangGraph và Chạy Pipeline

In [ ]:
def router_condition(state: AgentState):
    if state["question_type"] == "CODEABLE": return "coder"
    return "instructor"

def coder_condition(state: AgentState):
    if state.get("retry_count", 0) == 1 and not state.get("python_success", True):
        return "coder"
    return "instructor"

workflow = StateGraph(AgentState)
workflow.add_node("router", node_router)
workflow.add_node("coder", node_coder)
workflow.add_node("instructor", node_instructor)

workflow.set_entry_point("router")
workflow.add_conditional_edges("router", router_condition)
workflow.add_conditional_edges("coder", coder_condition)
workflow.add_edge("instructor", END)

app = workflow.compile()

import pandas as pd
from tqdm import tqdm

# ĐỌC FILE DỮ LIỆU ĐỂ TEST 
input_file = "test.json" 
output_file = "submission.csv"
test_qs = []

try:
    if input_file.endswith('.jsonl'):
        with open(input_file, 'r', encoding='utf-8') as f:
            test_qs = [json.loads(line) for line in f]
    elif input_file.endswith('.json'):
        with open(input_file, 'r', encoding='utf-8') as f:
            test_qs = json.load(f)
    print(f"Đã tải thành công {len(test_qs)} câu hỏi từ {input_file}!")
except Exception as e:
    print(f"Lỗi đọc file (Tạo dữ liệu mẫu): {e}")
    test_qs = [
        {"qid": "test_01", "question": "1 + 1 = ?", "choices": ["1", "2", "3", "4"]},
        {"qid": "test_02", "question": "Hà Nội nằm ở đâu?", "choices": ["Bắc", "Trung", "Nam"]}
    ]

# CHẠY SUY LUẬN TOÀN BỘ (BATCH INFERENCE)

import os

# Xóa file cũ nếu đã tồn tại
if os.path.exists(output_file):
    os.remove(output_file)

results = []
for q in tqdm(test_qs, desc="Đang suy luận"):
    state = {
        "qid": q.get("qid", ""),
        "question": q.get("question", ""),
        "choices": q.get("choices", []),
        "retry_count": 0
    }
    # Gọi luồng chạy của LangGraph
    res = app.invoke(state)
    
    # Tạo kết quả của câu hiện tại
    current_result = {
        "qid": state["qid"],
        "answer": res["final_answer"]
    }
    results.append(current_result)
    
    # LƯU TRỰC TIẾP VÀO CSV NGAY LẬP TỨC
    df = pd.DataFrame([current_result])
    df.to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)

print(f"\n✅ Đã chạy xong toàn bộ và lưu an toàn vào {output_file}.")

